# AI Wine Sommelier RAG

## Wine Review Indexing

https://www.kaggle.com/datasets/christopheiv/winemagdata130k

In [1]:
%pip install -Uqqq langchain langchain-community langchain-openai langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

### Pinecone 테스트


In [3]:
# 데이터로드
from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain은 LLM 기반 애플리케이션을 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://langchain.com/docs", "author": "alice", "page": 1}),
    Document(page_content="ChromaDB는 오픈소스 벡터 데이터베이스입니다.", metadata={"source": "https://chromadb.org/intro", "license": "MIT", "date": "2024-07-01"}),
    Document(page_content="파이썬으로 AI 서비스를 개발할 수 있습니다.", metadata={"source": "https://pythonai.co.kr", "editor": "kim", "page": 7}),
    Document(page_content="LLM은 자연어 처리를 위한 대형 언어 모델을 의미합니다.", metadata={"source": "https://llmwiki.com/info", "author": "bob", "version": "v1.1"}),
    Document(page_content="RAG는 검색과 생성의 결합 방식을 제공합니다.", metadata={"source": "https://rag-search.io", "reviewer": "lee", "section": "summary"}),
    Document(page_content="벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.", metadata={"source": "https://vectorbase.net", "author": "jin", "topic": "vector"}),
    Document(page_content="LangChain을 이용하면 다양한 AI 파이프라인을 구축할 수 있습니다.", metadata={"source": "https://langchain.com/blog", "editor": "sarah", "date": "2024-06-30"}),
    Document(page_content="OpenAI의 GPT 모델은 텍스트 생성에 특화되어 있습니다.", metadata={"source": "https://openai.com/gpt", "lang": "ko", "page": 5}),
    Document(page_content="파이썬은 AI 및 데이터 분석 분야에서 널리 사용되는 언어입니다.", metadata={"source": "https://python.org/usecases", "author": "chun", "updated": "2024-05"}),
    Document(page_content="Streamlit은 파이썬으로 대시보드를 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://streamlit.io/start", "editor": "park", "date": "2024-04-28"}),
    Document(page_content="Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.", metadata={"source": "https://retrieval.ai/dense", "type": "tech", "page": 3}),
    Document(page_content="Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.", metadata={"source": "https://pandas.pydata.org/about", "maintainer": "koh", "section": "intro"}),
    Document(page_content="메타데이터 필터링은 검색 결과의 품질을 높여줍니다.", metadata={"source": "https://search.com/metadata", "author": "seo", "feature": "filter"}),
    Document(page_content="SelfQueryRetriever는 자연어 쿼리를 임베딩 쿼리로 변환해줍니다.", metadata={"source": "https://selfquery.ai", "editor": "min", "date": "2024-05-12"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 극대화하는 방법입니다.", metadata={"source": "https://prompting.dev/guide", "author": "yang", "topic": "prompt"}),
    Document(page_content="HyDE 기법은 하이브리드 검색에 사용됩니다.", metadata={"source": "https://hyde-tech.com", "reviewer": "kang", "version": "2024.1"}),
    Document(page_content="CoT는 복잡한 문제를 단계적으로 해결하는 프롬프트 기법입니다.", metadata={"source": "https://cotprompt.org", "editor": "jung", "date": "2023-12-01"}),
    Document(page_content="문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.", metadata={"source": "https://embedding.ai/intro", "section": "embedding", "author": "song"}),
    Document(page_content="CrewAI는 멀티 에이전트 시스템 구현을 돕는 툴입니다.", metadata={"source": "https://crew.ai/docs", "lang": "ko", "page": 9}),
    Document(page_content="Fine-tuning은 사전학습 모델을 특정 도메인에 맞게 재학습시키는 과정입니다.", metadata={"source": "https://finetune.ai/guide", "editor": "jeon", "date": "2024-01-30"})
]

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # 1536차원 임베딩 모델

# 문서 -> 임베딩 -> Pinecone 업로드
vector_store = PineconeVectorStore.from_documents(
    documents,  # List[Document]
    embeddings, 
    index_name = 'pinecone-test'
)

In [5]:
# 코사인 유사도로 검색
retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 5}
)

retriever.invoke('벡터 데이터베이스란?')

[Document(id='c24f9144-108c-48e3-a780-4a537d90e9a8', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='8b1f9b11-d317-470c-8515-e5327e4ad3e2', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='0d308790-19c9-440c-b1a5-87fc81be0cd2', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='a95a2dd5-49b8-42c4-b419-caf60d0191f9', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.'),
 Document(id='c87b9f0e-d2be-4cb5-bd2e-d8a4da760012', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.')]

In [6]:
!gdown 1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3

Downloading...
From: https://drive.google.com/uc?id=1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3
To: c:\Users\SJ\OneDrive\Desktop\study\LLM\06_2stage_rag\winemag-data-130k-v2.csv

  0%|          | 0.00/52.9M [00:00<?, ?B/s]
  1%|          | 524k/52.9M [00:00<00:26, 1.94MB/s]
  3%|▎         | 1.57M/52.9M [00:00<00:11, 4.58MB/s]
  6%|▌         | 3.15M/52.9M [00:00<00:06, 7.86MB/s]
 10%|▉         | 5.24M/52.9M [00:00<00:04, 10.9MB/s]
 13%|█▎        | 6.82M/52.9M [00:00<00:03, 12.1MB/s]
 16%|█▌        | 8.39M/52.9M [00:00<00:03, 12.2MB/s]
 19%|█▉        | 9.96M/52.9M [00:00<00:03, 12.0MB/s]
 22%|██▏       | 11.5M/52.9M [00:01<00:03, 12.9MB/s]
 25%|██▍       | 13.1M/52.9M [00:01<00:03, 13.2MB/s]
 28%|██▊       | 14.7M/52.9M [00:01<00:02, 13.0MB/s]
 32%|███▏      | 16.8M/52.9M [00:01<00:02, 13.5MB/s]
 36%|███▌      | 18.9M/52.9M [00:01<00:02, 14.6MB/s]
 40%|███▉      | 21.0M/52.9M [00:01<00:02, 14.8MB/s]
 43%|████▎     | 22.5M/52.9M [00:01<00:02, 14.5MB/s]
 46%|████▌     | 24.1M/52.9M [00:01<00:01, 14.

In [7]:
# CSVLoader : CSV의 각 행을 하나의 Document로 변환하는 로더
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader('winemag-data-130k-v2.csv', encoding = 'utf-8')
docs = loader.load() # CSV를 List[Document]
print(len(docs))

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_78340\977801113.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


129971


In [8]:
for i, doc in enumerate(docs[:2]):
    print(f"{i}: {type(doc)}")
    print(f"{doc.metadata}")
    print(f"{doc.page_content}")
    print()

0: <class 'langchain_core.documents.base.Document'>
{'source': 'winemag-data-130k-v2.csv', 'row': 0}
: 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia

1: <class 'langchain_core.documents.base.Document'>
{'source': 'winemag-data-130k-v2.csv', 'row': 1}
: 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15.0
province: Douro
region_1: 
region_2: 
taster

In [10]:
# Pinecone 인덱스에 연결된 벡터스토어 객체 생성
vector_store = PineconeVectorStore(
    index_name='winemag-data',
    embedding=embeddings
)

batch_size = 100

for i in range(0, len(docs), batch_size):
    # docs에서 최대 100개씩 가져오기
    batch_data = docs[i:i + batch_size]

    # Document 임베딩 후 Pinecone에 업로드
    vector_store.add_documents(batch_data)

    print(f"index: {i} ~ {i + len(batch_data) - 1}")

index: 0 ~ 99
index: 100 ~ 199
index: 200 ~ 299
index: 300 ~ 399
index: 400 ~ 499
index: 500 ~ 599
index: 600 ~ 699
index: 700 ~ 799
index: 800 ~ 899
index: 900 ~ 999
index: 1000 ~ 1099
index: 1100 ~ 1199
index: 1200 ~ 1299
index: 1300 ~ 1399
index: 1400 ~ 1499
index: 1500 ~ 1599
index: 1600 ~ 1699
index: 1700 ~ 1799
index: 1800 ~ 1899
index: 1900 ~ 1999
index: 2000 ~ 2099
index: 2100 ~ 2199
index: 2200 ~ 2299
index: 2300 ~ 2399
index: 2400 ~ 2499
index: 2500 ~ 2599
index: 2600 ~ 2699
index: 2700 ~ 2799
index: 2800 ~ 2899
index: 2900 ~ 2999
index: 3000 ~ 3099
index: 3100 ~ 3199


KeyboardInterrupt: 

## Retrieval & Generation
1. 텍스트/이미지 입력으로 요리에 설명 chain
2. 요리설명텍스트 벡터db조회 chain
3. 요리설명/리뷰검색을 가지고 와인추천 응답 chain

### 요리설명 chain

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate
)
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


system_prompt = """
기존에 작성한 긴 시스템 프롬프트를 여기에 그대로 넣으세요.

주의사항:
맛에 대한 묘사만 줄글 형식으로 50자 이내로 작성하세요.
"""


def describe_dish_flavor(query: dict) -> str:
    temp = [
        {
            'text': '사용자가 제공한 이미지의 요리명과 풍미를 묘사해 주세요.'
        }
    ]

    if query.get('image_urls'):
        temp += [
            {
                'image_url': {
                    'url': image_url
                }
            }
            for image_url in query.get('image_urls')
        ]

    if query.get('text'):
        temp += [
            {
                'text': query.get('text')
            }
        ]

    prompt = ChatPromptTemplate.from_messages([
        ('system', system_prompt),
        HumanMessagePromptTemplate.from_template(temp)
    ])

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain.invoke({})

# 입력을 받아 그대로 Prompt에 전달해주는 chain을 실행할 수 있는 Runnable
dish_flavor_chain = RunnableLambda(describe_dish_flavor)

response = dish_flavor_chain.invoke({
    'text': '',
    'image_urls': [
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMjVfODIg%2FMDAxNzYxMzc2NjMzMDA4.-qxYpSDZfPleD8cj9VzxvqckYRIvaGpZW-fibT3whjsg.mefkB_k7NzVsXrb9RGPEQvZAplyzrustInLMV-827Gkg.JPEG%2FIMG%25A3%25DF2865.JPG&type=sc960_832',
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTA5MjBfMjEz%2FMDAxNzU4MzgwMDM4MzAz.XWaCn8Xu_7jjbYWq5P4MFqibaqNz4p3CFKRgjOnP2dMg.7wrfjF9U-p-CORf9ix4DbEGFRnOkaNh2ihjYlZOZy6Ag.JPEG%2FIMG_6083.JPG&type=sc960_832'
    ]
})

print(response)

바삭한 돈가스와 매콤달콤한 제육볶음 풍미가 어우러집니다.


### 리뷰 검색 chain

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 요리 풍미 설명을 받아서 Pinecone에서 유사한 와인리뷰를 찾아 반환하는 함수
def search_wine_review(query):
    embeddings = OpenAIEmbeddings(  
        model='text-embedding-3-small' 
    )  # 1536차원 임베딩 모델

    vector_store = PineconeVectorStore(
        index_name='winemag-data',  # 검색할 인덱스명
        embedding=embeddings        # 질의문 임베딩에 사용할 모델
    )

    docs = vector_store.similarity_search(
        query,
        k=5
    )

    return {
        'dish_flavor': query,
        'wine_reviews': '\n\n'.join(
            doc.page_content for doc in docs
        )
    }


query = '''
첫 번째 이미지는 '허브 그릴 스테이크'입니다. 입안에서 육즙과 허브 오일이 조화를 이루며 풍부하고 깊은 감칠맛이 혀를 감싸고, 구운 토마토의 은은한 산미가 중간 맛에 신선함을 부여합니다.
두 번째 이미지는 '시저 샐러드'입니다. 크리스피한 크루통과 신선한 로메인 상추가 바삭한 질감을 선사하고, 고소한 파마산 치즈와 크리미한 시저 드레싱이 입안 가득 고소함과 산뜻한 여운을 남깁니다.
'''

result = search_wine_review(query)
print(result['wine_reviews'])

: 2211
country: US
description: A strong note of grapefruit skin and juice, from ruby red to yellow, melds with crushed chalk and cider on the nose of this bottling. The citrus zest is prominent on the palate, as is more apple cider and a solid pithy grip, proving quite intriguing texturally.
designation: Mitzi
points: 88
price: 
province: California
region_1: Monterey County
region_2: Central Coast
taster_name: Matt Kettmann
taster_twitter_handle: @mattkettmann
title: Myka 2014 Mitzi Chardonnay (Monterey County)
variety: Chardonnay
winery: Myka

: 1603
country: Italy
description: Made with Nero d'Avola and aged in wood, this unusual rosato opens with aromas that recall mature red fruit, cake spice, chopped sage, oak and a balsamic note. The smooth, round palate offers candied red cherry, tangerine zest, toasted almond and a note of candied ginger alongside bright acidity. It has a tutti frutti finish but the overall array of flavors gives it some depth.
designation: Memorie
points: 88

In [ ]:
# 파이프라인 중간점검 (요리 풍미 추출 -> 와인 리뷰 검색)
search_wine_review_chain = RunnableleLambda(search_wine_review)

chain = dish_flavor_chain | search_wine_review_chain

response = chain.invoke({
    "text":"",
    "image_urls": [
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832',
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832'
    ]
})


{'dish_flavor': '허브 스테이크는 진하고 육즙 풍부하며, 시저샐러드는 고소하고 상큼하다.', 'wine_reviews': ": 433\ncountry: US\ndescription: Smoky aromas and a bold apricot flavor give this dry and seemingly light-bodied wine a distinct personality. Fermented in small concrete egg-shaped vats, it's far from simple and fruity, and instead offers great acidity and a tangy, tempting, appetizing personality.\ndesignation: Stepping Stones\npoints: 89\nprice: 27.0\nprovince: California\nregion_1: Santa Barbara County\nregion_2: Central Coast\ntaster_name: Jim Gordon\ntaster_twitter_handle: @gordone_cellars\ntitle: Prospect 772 2014 Stepping Stones Grenache Blanc (Santa Barbara County)\nvariety: Grenache Blanc\nwinery: Prospect 772\n\n: 456\ncountry: US\ndescription: Thickly moussed, Roederer's Brut, 60% Chardonnay blended with 40% Pinot Noir, aged two years on yeast is affordable and luxurious. A gorgeous, very dry sparkling wine with aromatics of honey, raisin, apple and pear, it is rich and soft and offers both depth and

In [23]:
response = chain.invoke({"text": '오늘 저녁은 버터와 허브로 구운 캐비어 가리비 관자 구이를 먹겠다.'})

print(response)

{'dish_flavor': '버터의 고소함과 허브 향이 어우러진 부드럽고 담백한 풍미가 느껴집니다.', 'wine_reviews': ": 351\ncountry: Hungary\ndescription: This amber-colored Hungarian stunner has ethereal aromas of raw honey, beeswax, freshly baked tarte Tatin, canned apricot and fresh pear. It is silky smooth and creamy in the mouth with pronounced flavors of clover honey, pear tart and baked sweet apples. The acidity is perfectly balanced so you can appreciate its velvety richness without feeling that the sweetness is too cloying.\ndesignation: Eszencia\npoints: 96\nprice: 320.0\nprovince: Tokaji\nregion_1: \nregion_2: \ntaster_name: Jeff Jenssen\ntaster_twitter_handle: @worldwineguys\ntitle: Oremus 2005 Eszencia  (Tokaji)\nvariety: Furmint\nwinery: Oremus\n\n: 1486\ncountry: Hungary\ndescription: The rich, spicy and savory nose evoke the food for which the country is known, and this wine would undoubtedly pair well with full-bodied, spiced meats and stews. Its full, spicy nose leads into dependable flavors of red berry, wo

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


# 와인 추천 체인: 요리 풍미와 검색된 와인 리뷰를 바탕으로 페어링 추천
def recommend_wines(query):
    prompt = ChatPromptTemplate.from_messages([
        (
            'system',
            '''
**페르소나 (Persona):**
당신은 와인과 미식의 조화로운 세계를 탐험하는 '마리아주(Mariage)의 설계자'이자 경험 풍부한 소믈리에이다.
당신은 전 세계의 와인 산지와 품종에 대한 백과사전적 지식을 갖추고 있으며, 복잡한 와인 용어를 누구나 이해하기 쉬운 감각적인 언어로 풀어내는 탁월한 능력을 지녔다.
당신의 태도는 언제나 환대하는 마음(Hospitality)으로 가득 차 있어, 와인 초보자부터 애호가까지 모두를 편안하게 이끈다.

**역할 (Role):**
당신의 가장 중요한 역할은 사용자가 준비한 요리에 '영혼의 단짝'이 될 와인을 추천하는 것이다.

1. 요리의 재료, 소스, 조리법을 분석한다.
2. 산도, 당도, 타닌, 바디감을 고려해 와인을 선정한다.
3. 와인이 음식과 어울리는 이유를 구체적으로 설명한다.

**가이드라인 (Guidelines):**

- 모든 답변은 구체적인 요리에 대한 와인 추천으로 구성한다.
- 와인이 음식의 풍미를 어떻게 증폭하거나 보완하는지 설명한다.
- 제공된 와인 리뷰 정보 안에서만 와인을 추천한다.
'''
        ),

        # 입력 변수(dish_flavor, wine_reviews) 기반 요청
        (
            'human',
            '''
와인 페어링 추천에 있어 아래에 제시된 요리와 풍미, 와인 리뷰만을 기초로 답변해 주세요.

## 요리와 풍미

{dish_flavor}

## 와인 리뷰 정보

{wine_reviews}
'''
        )
    ])

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain  # 체인 객체 반환


recommend_wines_chain = RunnableLambda(recommend_wines)

response = recommend_wines_chain.invoke({
    'dish_flavor': (
        '허브 스테이크는 겉은 바삭하고 속은 육즙이 가득하며, '
        '시저 샐러드는 고소하면서 상큼하다.'
    ),
    'wine_reviews': (
        'country: US\n'
        'description: The winery\'s annual stainless-steel bottling.'
    )
})

print(response)

### 추천: 미국산 연례 스테인리스 스틸 보틀링 와인

이 와인은 제공된 정보상 **스테인리스 스틸로 양조·숙성된 미국 와인**이라는 점만 확인되므로, 품종이나 타닌·바디감은 판단하기 어렵습니다. 다만 스테인리스 스틸 스타일은 일반적으로 오크 풍미를 덜어내고 와인의 신선한 인상을 살리는 방향이어서, **고소하면서 상큼한 시저 샐러드**와는 비교적 자연스럽게 어울릴 가능성이 있습니다. 샐러드의 산뜻함이 와인의 신선한 느낌을 돋보이게 하고, 드레싱의 고소함도 무겁게 덮지 않습니다.

허브 스테이크와의 조화는 조금 더 신중해야 합니다. 겉의 바삭한 풍미와 육즙에는 어느 정도의 바디감과 타닌이 필요하지만, 현재 리뷰에는 그 정보가 없습니다. 따라서 이 와인이 **가볍고 산뜻한 타입이라면 스테이크보다는 샐러드에 초점이 맞는 페어링**이 됩니다. 스테이크의 허브 향을 압도하지 않도록 **약간 차갑게, 10~12°C 정도**로 서빙해 보세요.  

결론적으로, 이 와인은 **허브 스테이크보다는 시저 샐러드와 더 안정적으로 어울리는 선택**이며, 스테이크와 함께할 경우에는 와인의 바디감이 충분한지 확인하는 것이 좋습니다.


### 통합 chain

In [ ]:
# {'text': ..., 'image_urls': ...} -> 요리 풍미(텍스트)
dish_flavor_chain = RunnableLambda(describe_dish_flavor)       
# 풍미 텍스트 -> 유사한 와인 리뷰 검색 -> {'dish_flavor': ..., 'wine_reviews':...}
search_wine_review_chain = RunnableLambda(search_wine_review) 
#{'dish_flavor: ..., 'wine_reviews':..., } 최종 와인 페어링 추천
recommend_wines_chain = RunnableLambda(recommend_wines)

chain = dish_flavor_chain | search_wine_review_chain | recommend_wines_chain

response = chain.invoke({
    'text': "",
    "image_urls": [
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%2866%29.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})

print(response)

## 추천: Laetitia 2008 Cuvée M Sparkling

**육즙 가득한 스테이크와 시저샐러드**에는 라에티시아의 **Cuvée M 스파클링**이 가장 잘 어울립니다.

- **스테이크와의 조화:** 부드럽고 크리미한 질감이 육즙의 풍성함을 자연스럽게 감싸며, 버터 토스트와 효모 풍미가 구운 고기의 고소하고 savory한 뉘앙스와 이어집니다.
- **시저샐러드와의 조화:** 라임과 귤의 산뜻한 풍미가 시저드레싱의 짭짤하고 상큼한 맛을 깨끗하게 정리합니다. 크리미한 질감은 드레싱의 고소함과도 조화롭습니다.
- **전체적인 균형:** 꿀처럼 은은한 단맛과 부드러운 거품이 스테이크의 진한 풍미를 압도하지 않으면서, 샐러드의 신선함까지 함께 살려줍니다.

### 서빙 팁
차갑게, 약 **8~10°C**로 준비하면 라임과 귤의 산뜻함이 더욱 살아납니다. 스테이크와 샐러드를 한 접시에 함께 즐길 때 특히 좋은 선택입니다.

**추천 와인:** Laetitia 2008 Cuvée M Sparkling  
**평점/가격:** 92점 / $35 굴절.


In [29]:
response = chain.invoke({
    'text': "이따 피자스쿨 베이컨포테이토 피자 먹을거다. 와인은 뭘 먹으면 좋을까?",
    "image_urls": [
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyMTExMjZfOTAg%2FMDAxNjM3ODc3MTI3MDY4.GrivVlwn1aHP8ydQZIA7mym_2AUzq7UaB8zqoTc1d8Mg.z6fe2yK3Tq9E95fHOpHJBq-c3fRUUTGPemiSY939QBcg.JPEG.kkuljo%2F20200406_210410.jpg&type=sc960_832"
    ]
})

print(response)

## 가장 추천: Laurent Gauthier 2013 Rosé — Beaujolais Rosé

짭짤한 베이컨과 감자의 조합에는 **Laurent Gauthier 2013 Rosé**가 가장 잘 어울립니다.

- **풍미의 연결:** 야생 딸기 풍미와 둥글고 과일감 있는 질감이 감자의 부드럽고 담백한 맛을 살려줍니다.
- **짠맛과의 균형:** 마지막에 느껴지는 **아주 선명한 산도**가 베이컨의 짭짤한 풍미를 산뜻하게 정리합니다.
- **질감의 조화:** 풍부한 스타일의 로제이면서도 지나치게 무겁지 않아, 베이컨과 감자의 맛을 덮지 않고 곁에서 받쳐줍니다.

차갑게 마시되 너무 차갑지 않게 준비하면, 베이컨의 짠맛과 감자의 포근함 사이에 산뜻한 균형을 만들어줄 것입니다.

### 대안

1. **Barton & Guestier 2011 Bistro Pinot Noir**  
   부드럽고 과일감이 있으며, 산도가 있어 가벼운 레드로 적합합니다. 감자의 둥근 맛에는 부드럽게 어울리고, 산도가 베이컨의 짠맛을 깔끔하게 다듬어줍니다.

2. **Le Petit Cochonnet 2015 Pinot Noir**  
   달콤한 체리, 라즈베리, 딸기 풍미가 명확하고 타닌이 거의 없어 편안합니다. 특히 무겁지 않은 베이컨·감자 요리에 부담 없이 곁들이기 좋습니다.

**결론적으로, 짭짤한 베이컨과 감자에는 Laurent Gauthier 로제를 1순위로, 가벼운 레드를 원한다면 Barton & Guestier Pinot Noir를 추천합니다.**


In [30]:
response = chain.invoke({
    'text': "와인 칵테일?"
})

print(response)

### 추천 와인: Fritz Haag 2014 Riesling (Mosel) — 91점, $22

달콤한 과일 향과 산뜻한 산미가 중심인 요리에는 **Fritz Haag 2014 Riesling**이 가장 잘 어울립니다. 이 와인은 **은은한 꿀·레몬의 단맛**이 있어 요리의 달콤한 과일 풍미를 자연스럽게 받아주고, **라임과 시트러스의 산미**가 입안을 산뜻하게 정리합니다.

특히 요리에 있는 은은한 와인 풍미를 압도하지 않으면서, 와인의 **싱그러운 허브와 라임 향**이 과일의 향을 더욱 또렷하게 살려줍니다. 약간의 잔당감과 강한 산미의 균형 덕분에 요리가 지나치게 달게 느껴지지 않는 것도 장점입니다.

**서빙 팁:** 차갑게, 약 8~10°C로 즐기면 과일 향과 산뜻한 마무리가 더욱 선명해집니다.

#### 대안
- **Blue Fish 2015 Original Riesling (Pfalz), 87점, $10**  
  살짝 드라이한 스타일로, 라즈베리와 화이트 피치 풍미가 요리의 달콤한 과일 향을 편안하게 강조합니다. 부담 없는 가격의 쉬운 페어링입니다.
- **Dr. Heidemanns-Bergweiler 2015 Dry Mineral Riesling (Mosel), 91점, $20**  
  요리가 덜 달고 산미가 더 두드러진다면 추천합니다. 자몽과 허니듀 풍미, 레몬 제스트의 산뜻한 쌉쌀함이 과일의 단맛을 깔끔하게 다듬어 줍니다.


In [32]:
response = chain.invoke({
    'text': "고추잡채"
    
})

print(response)

## 고추잡채에 가장 잘 맞는 와인

### 1순위: **Sipp Mack 2014 Rosacker Grand Cru Riesling (Alsace)**
- **가격:** $43  
- **평점:** 92점  
- **스타일:** 드라이한 리슬링

고추잡채의 **매콤달콤한 소스와 아삭한 피망**에는 이 와인의 레몬 같은 산도와 풍부한 사과 풍미가 잘 어울립니다. 드라이한 맛이 돼지고기의 담백함과 소스의 단맛을 무겁게 만들지 않고, 산뜻한 산도가 입안을 씻어내 다음 한입을 깔끔하게 준비합니다.

특히 신선한 빨강·노랑·초록 사과 풍미는 피망의 싱그러운 식감과 자연스럽게 연결되고, 구운 사과와 말린 사과의 뉘앙스는 돼지고기의 은은한 고소함을 보완합니다. **매운맛을 누르기보다 과일 풍미와 레몬 같은 상쾌함으로 균형을 잡아주는 선택**입니다.

### 대안

- **Kuentz-Bas 2014 Pfersigberg Grand Cru Riesling (Alsace)**  
  레몬 제스트, 신선한 사과, 사과 셔벗 같은 인상이 있어 고추잡채를 더욱 가볍고 산뜻하게 즐기기 좋습니다. 특히 피망의 아삭함과 매콤한 소스에 잘 맞습니다.

- **Wittmann 2015 Trocken Scheurebe (Rheinhessen)**  
  복숭아와 자몽의 풍미, 라임과 미네랄감이 뚜렷합니다. 고추잡채의 달콤한 소스에는 복숭아 풍미가, 매콤함에는 라임과 자몽의 상쾌함이 대응합니다. 리슬링보다 조금 더 향긋하고 개성 있는 방향입니다.

**최종 추천:** 고추잡채의 매콤달콤함과 돼지고기, 피망을 가장 균형 있게 아우르는 **Sipp Mack 2014 Rosacker Grand Cru Riesling**입니다.
